<a href="https://colab.research.google.com/github/adenikeadewumi/Python-programming-for-ML-WIEOAU/blob/main/09_advanced_python/exercises/09_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Solutions — Module 09: Advanced Python

---

### Exercise 1 — Primes below 100 using list comprehension

**Concept:** A number is prime if it is only divisible by 1 and itself. We check divisibility for all numbers from 2 up to sqrt(n) — no need to check higher. A helper function inside the comprehension keeps it readable.

In [ ]:
def is_prime(n):
    """Return True if n is a prime number."""
    if n < 2:
        return False
    for i in range(2, int(n**0.5) + 1):   # only need to check up to sqrt(n)
        if n % i == 0:
            return False
    return True

primes = [n for n in range(2, 100) if is_prime(n)]
print(f"Primes below 100 ({len(primes)} total):")
print(primes)

# Why check up to sqrt(n)? If n has a factor larger than sqrt(n),
# it must also have a factor smaller than sqrt(n) — so we'd have found it already.

### Exercise 2 — Running average generator

**Concept:** A generator uses `yield` to produce values lazily. Each call to `next()` resumes from where it left off. This is perfect for computing a running total/average as numbers arrive.

In [ ]:
def running_average(numbers):
    """
    Yield the running average as each number is added.
    
    Example: [10, 20, 30] yields 10.0, 15.0, 20.0
    """
    total = 0
    count = 0
    
    for num in numbers:
        total += num
        count += 1
        yield total / count   # pause here, give the caller the current average

# Test
data = [10, 20, 30, 40, 50]
gen = running_average(data)

for i, avg in enumerate(running_average(data), 1):
    print(f"After {i} numbers: running avg = {avg:.1f}")

# Real use: streaming data — you don't need all values in memory at once

**Why generators?** If you are processing a sensor stream or a huge file, you cannot load everything into memory. A generator computes on demand — it only holds the current number in memory, not the whole list.

### Exercise 3 — SuppressErrors context manager

**Concept:** A context manager has two phases: setup (before `yield`) and teardown (after `yield`). The `@contextmanager` decorator lets you write this as a generator function. Catching exceptions inside the `with` block is done via try/except around `yield`.

In [ ]:
from contextlib import contextmanager

@contextmanager
def suppress_errors(*exception_types):
    """
    Context manager that silently catches specified exceptions.
    
    Usage:
        with suppress_errors(ValueError, KeyError):
            risky_code()
    """
    try:
        yield   # hand control to the 'with' block
    except exception_types:
        pass    # silently swallow the exception

# Tests
print("Test 1: division by zero (suppressed)")
with suppress_errors(ZeroDivisionError):
    result = 1 / 0   # would normally crash
    print("This line never runs")
print("Execution continues after the with block")

print()
print("Test 2: key error (suppressed)")
d = {"a": 1}
with suppress_errors(KeyError):
    print(d["nonexistent"])
print("Still running!")

print()
print("Test 3: wrong type NOT suppressed")
try:
    with suppress_errors(ValueError):
        int("hello")   # ValueError — suppressed
        result = 1/0   # ZeroDivisionError — NOT suppressed, will raise
except ZeroDivisionError as e:
    print(f"ZeroDivisionError was not suppressed: {e}")

### Exercise 4 — Top-5 most common words

**Concept:** `collections.Counter` is a dict subclass specialised for counting. `.most_common(n)` returns the n most frequent items. We clean the text by converting to lowercase and stripping punctuation.

In [ ]:
from collections import Counter
import re   # regular expressions for cleaning

passage = '''
To be or not to be that is the question whether tis nobler in the mind
to suffer the slings and arrows of outrageous fortune or to take arms against
a sea of troubles and by opposing end them
'''

# Clean: lowercase, remove punctuation, split into words
words = re.findall(r'[a-z]+', passage.lower())   # keep only letter sequences

counter = Counter(words)
top5 = counter.most_common(5)

print("Top 5 most common words:")
for word, count in top5:
    bar = '#' * count
    print(f"  {word:<12} {count:2}  {bar}")

print()
print(f"Total words: {sum(counter.values())}")
print(f"Unique words: {len(counter)}")

**`re.findall(r'[a-z]+', text)`:** The pattern `[a-z]+` matches one or more lowercase letters. `findall` returns a list of all matches. This elegantly handles punctuation like commas and periods — they simply don't match the pattern and are skipped.

---